# Experiment 10 — K-Medoids clustering

## Run directly in Jupyter

- Keep the extracted `datasets` folder beside the `notebooks` folder.
- Run every cell from top to bottom with **Shift + Enter**, or choose **Run → Run All Cells**.
- No manual input or internet download is required.
- If imports are missing, run `%pip install numpy pandas matplotlib seaborn scikit-learn` in a code cell, then rerun the failed cell.

**Dataset:** `datasets/sample_titanic.csv` is an included synthetic teaching dataset, not the official Titanic passenger dataset. Scores are illustrative.

In [ ]:
# Load the included sample dataset. No internet or manual input is needed.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

possible_paths = [
    Path("sample_titanic.csv"),
    Path("../datasets/sample_titanic.csv"),
    Path("datasets/sample_titanic.csv"),
]
csv_file = next((p for p in possible_paths if p.exists()), None)
if csv_file is None:
    raise FileNotFoundError(
        "sample_titanic.csv not found. Keep the datasets folder beside notebooks."
    )

df = pd.read_csv(csv_file)
df = df.rename(columns={
    "PassengerId": "passenger_id", "Survived": "survived", "Pclass": "pclass",
    "Name": "name", "Sex": "sex", "Age": "age", "SibSp": "sibsp",
    "Parch": "parch", "Fare": "fare", "Embarked": "embarked", "Cabin": "cabin"
})
print("Loaded sample dataset:", csv_file)
print("Dataset shape:", df.shape)
display(df.head())


## Aim
Apply K-Medoids. Unlike K-Means, each medoid is an actual observation.

## 1. Install the additional dependency
Run this cell first. `%pip` installs into the Python environment used by the notebook. If it fails because of Python/NumPy compatibility, do not replace K-Medoids with K-Means.

In [ ]:
%pip install scikit-learn-extra


## 2. Select, clean and scale features

In [ ]:
features=["pclass","age","sibsp","parch","fare"]
X=df[features].copy()
for col in features: X[col]=X[col].fillna(X[col].median())
from sklearn.preprocessing import StandardScaler
X_scaled=StandardScaler().fit_transform(X)
print("Dataset shape:",X.shape)


## 3. Fit K-Medoids

In [ ]:
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score
model=KMedoids(n_clusters=3,metric="euclidean",method="pam",random_state=42)
labels=model.fit_predict(X_scaled)
df["KMedoids_Cluster"]=labels
print("Inertia:",round(model.inertia_,3))
print("Silhouette score:",round(silhouette_score(X_scaled,labels),4))
print("\nMedoids in original feature units:")
display(df.iloc[model.medoid_indices_][features])
print("\nCluster counts:\n",pd.Series(labels).value_counts().sort_index())


## 4. Plot clusters and medoids

In [ ]:
medoids=df.iloc[model.medoid_indices_]
plt.figure(figsize=(8,5))
plt.scatter(X["age"],X["fare"],c=labels,cmap="viridis",s=25,alpha=0.75)
plt.scatter(medoids["age"],medoids["fare"],marker="X",s=220,edgecolors="black",label="Medoids")
plt.xlabel("Age"); plt.ylabel("Fare"); plt.title("K-Medoids Clustering (K=3)")
plt.legend(); plt.show()


## Expected output
Inertia, silhouette score, medoid rows, cluster counts and scatter plot. K-Medoids may be slower than K-Means.